# Design patterns casebook (Module D)

In Lecture 20 we went through six design patterns with small examples. Small examples are easy to read, but they hide why anyone would bother: with ten lines of code, any structure works. The value of a pattern only shows up when the code is long enough to hurt and someone asks for a change.

So this notebook takes five patterns and shows each one on a longer, more realistic script. Most of the "before" scripts are based on my own research code, moved onto the course data. I wrote them, they worked, and then someone (usually me) needed a change.

## How each case works

1. **Before**: the script as it grew.
2. **The change request**: what someone asks for, and why it hurts in the "before" version.
3. **After**: the refactor, checked against the "before" results.
4. **The change again**: the same request on the refactored code.
5. **Discussion**: why this pattern and not a similar one, when it is not worth it, and where you have already used it in a library.

Each case ends with an exercise. **Use the empty cell above each solution to try it yourself first**, then run the solution. Solutions end with an `assert` that passes silently when the answer is right.

The cases stand on their own, so you can do them in any order. Run this setup cell first.

In [ ]:
import functools
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

# the course datasets (same files used in the lectures)
taxi = pd.read_csv("../data/2019_Yellow_Taxi_Trip_Data.csv")
tas = xr.open_dataset("../data/global_temparature.nc")["tas"].load()   # warming, time x lat x lon

def area_mean(da):
    # area-weighted mean over lat/lon (grid cells get smaller towards the poles)
    w = np.cos(np.deg2rad(da.lat))
    return da.weighted(w).mean(["lat", "lon"])

tmp = Path(tempfile.mkdtemp(prefix="casebook_"))   # scratch space for files we write
print("taxi:", taxi.shape, "| tas:", dict(tas.sizes))

# Case 1. Strategy: scoring forecasts with several metrics

Let's say we have three simple forecasts of the late-century (2081-2100) warming map, made only from data up to 2020:

 - **persistence**: assume the 2001-2020 map just stays as it is,
 - **trend**: fit a straight line at each grid point over 1971-2020 and extend it to 2090,
 - **pattern x 2**: keep the 2001-2020 *pattern*, scaled so the global mean is 2°C.

We want to score each one against what actually happens in the data, using a few metrics.

In [ ]:
truth = tas.sel(time=slice(2081, 2100)).mean("time")
recent = tas.sel(time=slice(2001, 2020)).mean("time")
fit = tas.sel(time=slice(1971, 2020)).polyfit("time", deg=1).polyfit_coefficients
forecasts = {
    "persistence": recent,
    "trend": fit.sel(degree=0) + fit.sel(degree=1) * 2090,
    "pattern x 2": recent / area_mean(recent) * 2.0,
}
print({name: round(float(area_mean(fc)), 2) for name, fc in forecasts.items()},
      "| truth:", round(float(area_mean(truth)), 2))

## Before

This is how my version grew, when I was comparing climate emulators for a paper. I wrote the RMSE block first, then copied it for the next metric, and the next. The real notebook had five of these cells, and the comment on the fourth one still said "Weighted R2".

In [ ]:
w = np.cos(np.deg2rad(truth.lat))

# Weighted RMSE of the late-century map
rmse = {}
for name, fc in forecasts.items():
    err = (fc - truth) ** 2
    rmse[name] = float(np.sqrt(err.weighted(w).mean()))
best_rmse = min(rmse, key=rmse.get)

# Weighted RMSE of the late-century map
mae = {}
for name, fc in forecasts.items():
    err = abs(fc - truth)
    mae[name] = float(err.weighted(w).mean())
best_mae = min(mae, key=mae.get)

# Weighted R2
r2 = {}
for name, fc in forecasts.items():
    ss_res = ((fc - truth) ** 2).weighted(w).sum()
    ss_tot = ((truth - truth.weighted(w).mean()) ** 2).weighted(w).sum()
    r2[name] = float(1 - ss_res / ss_tot)
best_r2 = max(r2, key=r2.get)               # higher is better for this one

# Global mean bias
bias = {}
for name, fc in forecasts.items():
    bias[name] = float((fc - truth).weighted(w).mean())
best_bias = min(bias, key=lambda k: abs(bias[k]))

before = pd.DataFrame({"rmse": rmse, "mae": mae, "r2": r2, "bias": bias})
print(before.round(3))
print("best:", {"rmse": best_rmse, "mae": best_mae, "r2": best_r2, "bias": best_bias})

## The change request

Two requests came in:

 1. "Can you also score just the Arctic (north of 60°N)?"
 2. "Add the correlation between the maps, using `scipy.stats.pearsonr`."

For the first one you have to edit all four blocks, the weights and `truth` included, and it is easy to miss one. For the second you copy a block a fifth time, and then work out how to fit `pearsonr` in, since it wants two flat arrays and returns a result object rather than a number.

The thing that varies here is *how we score*. Everything else (loop over forecasts, weight by area, pick the best) is the same in every block. That is what Strategy is for: pull out the part that varies into something you can pass in, and write the rest once.

## After

In [ ]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class Metric:
    name: str
    fn: Callable                      # fn(forecast, truth, weights) -> float
    higher_is_better: bool = False

def rmse(fc, truth, w):
    return float(np.sqrt(((fc - truth) ** 2).weighted(w).mean()))

def mae(fc, truth, w):
    return float(abs(fc - truth).weighted(w).mean())

def r2(fc, truth, w):
    ss_res = ((fc - truth) ** 2).weighted(w).sum()
    ss_tot = ((truth - truth.weighted(w).mean()) ** 2).weighted(w).sum()
    return float(1 - ss_res / ss_tot)

def abs_bias(fc, truth, w):
    return abs(float((fc - truth).weighted(w).mean()))

METRICS = [Metric("rmse", rmse), Metric("mae", mae),
           Metric("r2", r2, higher_is_better=True), Metric("abs bias", abs_bias)]

def score(forecasts, truth, metrics, region=None):
    # Score every forecast with every metric. `region` is an optional .sel() dict.
    region = region or {}
    truth = truth.sel(**region)
    w = np.cos(np.deg2rad(truth.lat))
    table = pd.DataFrame({
        m.name: {name: m.fn(fc.sel(**region), truth, w) for name, fc in forecasts.items()}
        for m in metrics
    })
    best = {m.name: table[m.name].idxmax() if m.higher_is_better else table[m.name].idxmin()
            for m in metrics}
    return table, best

after, best = score(forecasts, truth, METRICS)
print(after.round(3))
print("best:", best)

# same numbers as the copy-pasted version
assert np.allclose(after["rmse"], before["rmse"]) and np.allclose(after["r2"], before["r2"])
assert np.allclose(after["abs bias"], before["bias"].abs())

The scoring loop, the weights and the "pick the best" logic now exist once. Each metric is a small function, plus one fact the loop needs to know about it: whether higher is better. That fact is why `Metric` is a small class rather than a bare function. In my original notebook it lived in a different place in each cell (`highlight_max` in some, `highlight_min` in others).

## The change again

The Arctic is now an argument. For `pearsonr`, we write a small **adapter** so that a plain "two arrays in, number out" function fits our `fn(forecast, truth, weights)` interface:

In [ ]:
from scipy.stats import pearsonr

def from_arrays(fn):
    # Adapter: make an (a, b) -> number function fit our metric interface.
    # Note, it ignores the area weights, which is fine for a quick look.
    def metric(fc, truth, w):
        return float(fn(fc.values.ravel(), truth.values.ravel()))
    return metric

pearson = Metric("pearson r", from_arrays(lambda a, b: pearsonr(a, b).statistic),
                 higher_is_better=True)

arctic, best = score(forecasts, truth, METRICS + [pearson], region={"lat": slice(60, 90)})
print(arctic.round(3))
print("best in the Arctic:", best)

## Discussion

**Why Strategy and not Adapter?** In the lecture quiz most of the class picked Adapter for the nearest-neighbour version of this question, so it is worth spelling out. Both appear above, and they do different jobs:

 - **Strategy** is about *choice*. There are several interchangeable ways to do one step (RMSE, MAE, R², ...), and the code that uses them should not care which one it gets. `score()` is written once and takes the metric as an argument.
 - **Adapter** is about *fit*. You already have something that does the job (`pearsonr`), but its interface does not match the one your code expects, so you wrap it. You only need an adapter when a piece does not fit; RMSE and MAE did not need one because we wrote them to the interface in the first place.

So "swap distance metrics without touching the search" is Strategy. "Use scipy's distance function, which takes its arguments differently from ours" is Adapter. Usually you meet them together, like here: an adapter is how an outside function becomes one of your strategies.

Note, `from_arrays` is a function that wraps another function, which is also what a decorator does. The difference is the job again: an adapter *changes* the interface, a decorator keeps it the same and adds behaviour (Case 5).

**When it is not worth it:** if you only ever compute RMSE, a single function is fine. The second metric is when to start thinking about it, and by the third copy-pasted block it is overdue.

**You have already used this:**

 - `sorted(trips, key=lambda t: t["fare"])`: the sort is fixed, the comparison key is the strategy.
 - `df.groupby("zone").agg(["mean", "max"])` and `df.agg(my_function)`.
 - `scipy.spatial.distance.cdist(a, b, metric="cityblock")`, which also accepts your own function as the metric.
 - `sklearn.neighbors.KNeighborsClassifier(metric=...)`, which is the nearest-neighbour question from the lecture.

In [ ]:
from scipy.spatial.distance import cdist

a = np.array([[0.0, 0.0], [3.0, 4.0]])
b = np.array([[0.0, 0.0]])
print(cdist(a, b, metric="euclidean").ravel())                         # built-in strategy
print(cdist(a, b, metric=lambda u, v: np.abs(u - v).max()).ravel())    # our own strategy

### Exercise 1 - the nearest-neighbour question

Here is a nearest-neighbour search over the grid points of the climate data, written the way the scoring code started out: an `if`/`elif` on the metric name inside the search.

The search gives the wrong answer near the date line. A point at longitude 359° should be right next to the grid cells at longitude 0°, but in plain degrees they look 359° apart. That is the latitude/longitude trap from Lecture 16.

Refactor it so the search takes a **distance function** as an argument, then add a great-circle (haversine) distance and use it to find the 4 nearest grid cells to `(70, 359)`. Store the result as `near`, a list of `(lat, lon)` tuples.

In [ ]:
# grid points as an (n, 2) array of (lat, lon) in degrees
lat2d, lon2d = np.meshgrid(tas.lat.values, tas.lon.values, indexing="ij")
grid = np.column_stack([lat2d.ravel(), lon2d.ravel()])

def nearest_before(point, points, k, metric="euclidean"):
    if metric == "euclidean":
        d = np.sqrt(((points - point) ** 2).sum(axis=1))
    elif metric == "manhattan":
        d = np.abs(points - point).sum(axis=1)
    else:
        raise ValueError(metric)
    idx = np.argsort(d)[:k]
    return [(round(float(la), 2), float(lo)) for la, lo in points[idx]]

print(nearest_before(np.array([70.0, 359.0]), grid, k=4))

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
def nearest(point, points, k, distance):
    d = distance(point, points)
    idx = np.argsort(d)[:k]
    return [(round(float(la), 2), float(lo)) for la, lo in points[idx]]

def euclidean(p, pts):
    return np.sqrt(((pts - p) ** 2).sum(axis=1))

def haversine(p, pts, radius=6371.0):
    lat1, lon1 = np.deg2rad(p)
    lat2, lon2 = np.deg2rad(pts[:, 0]), np.deg2rad(pts[:, 1])
    h = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * radius * np.arcsin(np.sqrt(h))     # km

point = np.array([70.0, 359.0])
near = nearest(point, grid, k=4, distance=haversine)
print("great circle:", near)
print("euclidean:   ", nearest(point, grid, k=4, distance=euclidean))

assert nearest(point, grid, 4, euclidean) == nearest_before(point, grid, 4)   # same as before
assert any(lon == 0.0 for _, lon in near)        # great circle finds the cells across 0°
print("check passed")

# Case 2. Factory: one series, many sources

Global temperature records come from several groups, and each one publishes in its own format: different file types, header lines, units and baseline periods. For a paper I needed to load six of them into the same shape, and my loader grew one `if` per dataset.

Here we fake three sources from the course data, so the notebook runs anywhere. In real life these would be downloads from three different groups.

In [ ]:
gmst = area_mean(tas).to_series()          # the "true" global mean series, by year

# a gridded NetCDF file (like a climate model or reanalysis)
tas.to_netcdf(tmp / "gridded.nc")

# a CSV with three lines of notes at the top, relative to a 1961-1990 baseline
with open(tmp / "station.csv", "w") as f:
    f.write("# Station-based global mean\n# Anomalies relative to 1961-1990\n# Contact: someone@example.org\n")
    (gmst - gmst.loc[1961:1990].mean()).rename("Anomaly (C)").rename_axis("Year").to_csv(f)

# a whitespace-separated text file, no header, in Fahrenheit, relative to 1951-1980
((gmst - gmst.loc[1951:1980].mean()) * 9 / 5).round(4).to_csv(tmp / "us.txt", sep=" ", header=False)

print(sorted(p.name for p in tmp.iterdir()))

## Before

Like mine, it uses separate `if`s, so asking for a name it does not know crashes with an `UnboundLocalError` rather than a useful message. The baseline is fixed for one source and forgotten for another, and the plotting code further down needs its own `if` on the name to get the units right.

In [ ]:
def get_gmst(dataset):
    if dataset == "gridded":
        da = xr.open_dataarray(tmp / "gridded.nc")
        w = np.cos(np.deg2rad(da.lat))
        series = da.weighted(w).mean(["lat", "lon"]).to_series()
    if dataset == "station":
        df = pd.read_csv(tmp / "station.csv", skiprows=3)
        series = df.set_index("Year")["Anomaly (C)"]
    if dataset == "us":
        df = pd.read_csv(tmp / "us.txt", sep=r"\s+", names=["year", "anom_f"])
        series = df.set_index("year")["anom_f"] * 5 / 9
        series = series - series.loc[1851:1900].mean()     # put it on the same baseline
    return series

before = pd.DataFrame({name: get_gmst(name) for name in ["gridded", "station", "us"]})
print(before.loc[[1900, 2000, 2100]].round(2))
print("largest disagreement between sources:", round(float((before.max(axis=1) - before.min(axis=1)).max()), 2), "°C")

The three sources should agree, since they are the same data underneath, but they are about 0.1°C apart, because only the `us` branch puts its series on the common baseline. Small enough to look plausible on a plot, which is how it went unnoticed in my version for a while.

## The change request

 1. "Every source should come back on the same 1851-1900 baseline, with a name."
 2. "Add a fourth source." (Later a fifth, and a sixth.)

The first means editing every branch and every place the branches are used. The second means another branch, set up slightly differently from the others.

The thing that varies here is *which object we build and how we set it up*: the file name, the reader, the header lines, the column, the units. A **factory** puts that set-up in one place and hands back an object with the same interface whichever source you asked for.

## After

In [ ]:
class GriddedSource:
    def __init__(self, path, variable="tas"):
        self.path, self.variable = path, variable
    def load(self):
        return area_mean(xr.open_dataset(self.path)[self.variable]).to_series()

class CsvSource:
    def __init__(self, path, column, skiprows=0, index="Year"):
        self.path, self.column, self.skiprows, self.index = path, column, skiprows, index
    def load(self):
        return pd.read_csv(self.path, skiprows=self.skiprows).set_index(self.index)[self.column]

class FahrenheitTextSource:
    def __init__(self, path):
        self.path = path
    def load(self):
        df = pd.read_csv(self.path, sep=r"\s+", names=["year", "anom_f"])
        return df.set_index("year")["anom_f"] * 5 / 9        # a temperature difference, so no -32

SOURCES = {
    "gridded": lambda: GriddedSource(tmp / "gridded.nc"),
    "station": lambda: CsvSource(tmp / "station.csv", column="Anomaly (C)", skiprows=3),
    "us":      lambda: FahrenheitTextSource(tmp / "us.txt"),
}

def make_source(name):
    # Factory: build the right source object for a dataset name.
    try:
        return SOURCES[name]()
    except KeyError:
        raise ValueError(f"unknown dataset {name!r}, choose from {sorted(SOURCES)}") from None

def rebaseline(series, start=1851, end=1900):
    return series - series.loc[start:end].mean()

def get_gmst(name):
    return rebaseline(make_source(name).load()).rename(name)

after = pd.concat([get_gmst(name) for name in SOURCES], axis=1)
print(after.loc[[1900, 2000, 2100]].round(2))
spread = float((after.max(axis=1) - after.min(axis=1)).max())
print("largest disagreement between sources:", round(spread, 4), "°C")
assert spread < 0.01

Asking for a source that does not exist now gives a message that says what to do:

In [ ]:
try:
    get_gmst("berkeley")
except ValueError as e:
    print("ValueError:", e)

## The change again

The baseline is fixed in one place (`rebaseline`), for every source at once. A new source is one new entry in `SOURCES`, and a new class only if its format is new. Nothing that *uses* the series changes.

## Discussion

**Why Factory and not Strategy?** Both choose between alternatives, so they are easy to mix up. The difference is what gets chosen and who chooses.

 - With **Strategy** the caller chooses a *behaviour* and passes it in: "score with RMSE".
 - With a **factory** the caller asks for a *thing* by name (or by file extension, or from a config file), and the factory decides which class to build and how to set it up. The caller never sees `skiprows=3` or the Fahrenheit conversion.

A good hint is the word "load" or "make" or "open": if the alternatives need different *set-up* before they can do anything, that is a factory.

**When it is not worth it:** if every source is a CSV with the same columns, `pd.read_csv(path)` is your factory already. It starts to pay off at the second format, and certainly by the time you have one `if` per dataset.

**You have already used this:**

 - `xr.open_dataset(path, engine="netcdf4")`: xarray picks a backend from the engine name or the file.
 - The `pd.read_*` family, and `fsspec.open("s3://...")`, which picks a filesystem from the URL.
 - `np.random.default_rng(seed)` and `matplotlib.colormaps["viridis"]`: ask by name or seed, get a ready-to-use object.

### Exercise 2 - add a source

A fourth group publishes JSON, already in °C but relative to 1981-2010. The cell below writes it. Add it to the factory (a new class and one new entry in `SOURCES`) so that `get_gmst("json")` agrees with the other three to within 0.01°C.

In [ ]:
import json
payload = {"units": "degC", "baseline": "1981-2010",
           "data": {str(k): v for k, v in (gmst - gmst.loc[1981:2010].mean()).round(4).items()}}
(tmp / "anomalies.json").write_text(json.dumps(payload))
print(list(payload["data"].items())[:3])

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
class JsonSource:
    def __init__(self, path):
        self.path = path
    def load(self):
        data = json.loads(Path(self.path).read_text())["data"]
        return pd.Series({int(year): value for year, value in data.items()})

SOURCES["json"] = lambda: JsonSource(tmp / "anomalies.json")

diff = (get_gmst("json") - get_gmst("gridded")).abs().max()
print("max difference from gridded:", round(float(diff), 4))
assert diff < 0.01
print("check passed")

# Case 3. Observer: a fitting loop that does too much

Any long loop (training a model, running a calibration, processing hundreds of files) tends to collect extra jobs: print the progress, keep a history, save a checkpoint now and then, stop early if nothing is improving. Each one is small, so each one goes straight into the loop.

Here is a small gradient-descent fit of `tip = a * fare + b` on the taxi data, written that way. My own version was a loop over climate model runs, which had picked up skip-if-already-done checks, progress prints, `try`/`except` with "Skipping...", and printing the whole dataset before and after resampling.

## Before

In [ ]:
ok = taxi[(taxi["fare_amount"] > 0) & (taxi["fare_amount"] < 100)]
x = ((ok["fare_amount"] - ok["fare_amount"].mean()) / ok["fare_amount"].std()).values
y = ok["tip_amount"].values

def fit_tips(x, y, lr=0.1, n_steps=1000, verbose=True, print_every=100,
             keep_history=True, checkpoint_every=None, patience=None, tol=1e-6):
    a, b = 0.0, 0.0
    history, checkpoints = [], []
    best_loss, bad_steps = np.inf, 0
    for step in range(n_steps):
        pred = a * x + b
        loss = np.mean((pred - y) ** 2)
        a -= lr * 2 * np.mean((pred - y) * x)
        b -= lr * 2 * np.mean(pred - y)

        if keep_history:
            history.append(loss)
        if verbose and step % print_every == 0:
            print(f"step {step:4d}  loss {loss:.4f}")
        if checkpoint_every is not None and step % checkpoint_every == 0:
            checkpoints.append((step, a, b))          # in real life, a file on disk
        if patience is not None:
            if loss < best_loss - tol:
                best_loss, bad_steps = loss, 0
            else:
                bad_steps += 1
            if bad_steps >= patience:
                if verbose:
                    print(f"stopping early at step {step}")
                break
    return a, b, history, checkpoints

a, b, history, checkpoints = fit_tips(x, y, print_every=10, checkpoint_every=10, patience=5)
print(f"a = {a:.3f}, b = {b:.3f}, {len(history)} steps, {len(checkpoints)} checkpoints")

## The change request

 1. "Write the progress to a log file instead of the screen."
 2. "Stop straight away if the loss goes to `inf` or `nan`."

Each request adds another argument to `fit_tips` and another `if` inside the loop. The function already has nine arguments, and only three of them have anything to do with fitting. Testing it means testing every combination of flags, and the fitting maths is buried among them.

The thing that varies here is *what should happen after each step*. The loop itself is fixed. With **Observer**, the loop just announces "step done, here is the loss" to a list of listeners (usually called callbacks), and each listener decides what to do with it.

## After

In [ ]:
class Stop(Exception):
    # A callback raises this to end the fit early.
    pass

def fit_tips(x, y, lr=0.1, n_steps=1000, callbacks=()):
    a, b = 0.0, 0.0
    for step in range(n_steps):
        pred = a * x + b
        loss = np.mean((pred - y) ** 2)
        a -= lr * 2 * np.mean((pred - y) * x)
        b -= lr * 2 * np.mean(pred - y)
        try:
            for cb in callbacks:
                cb(step, loss, a, b)           # tell every listener
        except Stop as reason:
            print(f"stopped at step {step}: {reason}")
            break
    return a, b

# each listener is a small class, because each one keeps a bit of state
class History:
    def __init__(self):
        self.losses = []
    def __call__(self, step, loss, a, b):
        self.losses.append(loss)

class PrintProgress:
    def __init__(self, every=100):
        self.every = every
    def __call__(self, step, loss, a, b):
        if step % self.every == 0:
            print(f"step {step:4d}  loss {loss:.4f}")

class Checkpoint:
    def __init__(self, every):
        self.every, self.saved = every, []
    def __call__(self, step, loss, a, b):
        if step % self.every == 0:
            self.saved.append((step, a, b))

class EarlyStopping:
    def __init__(self, patience, tol=1e-6):
        self.patience, self.tol = patience, tol
        self.best, self.bad_steps = np.inf, 0
    def __call__(self, step, loss, a, b):
        if loss < self.best - self.tol:
            self.best, self.bad_steps = loss, 0
        else:
            self.bad_steps += 1
        if self.bad_steps >= self.patience:
            raise Stop(f"no improvement for {self.patience} steps")

hist, ckpt = History(), Checkpoint(every=10)
a2, b2 = fit_tips(x, y, callbacks=[hist, PrintProgress(every=10), ckpt, EarlyStopping(patience=5)])
print(f"a = {a2:.3f}, b = {b2:.3f}, {len(hist.losses)} steps, {len(ckpt.saved)} checkpoints")

assert np.isclose(a2, a) and np.isclose(b2, b) and len(hist.losses) == len(history)
assert len(ckpt.saved) == len(checkpoints)

Same fit, same history, same checkpoints, and `fit_tips` is back to three arguments that are about fitting.

## The change again

Logging to a file is one new listener, and the fit does not change at all:

In [ ]:
class LogToFile:
    def __init__(self, path, every=100):
        self.path, self.every = path, every
        Path(path).write_text("")              # start a fresh log
    def __call__(self, step, loss, a, b):
        if step % self.every == 0:
            with open(self.path, "a") as f:
                f.write(f"{step},{loss:.6f}\n")

fit_tips(x, y, callbacks=[LogToFile(tmp / "fit.log", every=10), EarlyStopping(patience=5)])
print((tmp / "fit.log").read_text())

## Discussion

**Why Observer and not Strategy?** Both pass functions in. A strategy is *one* function that decides how the work is done, so swapping it changes the result. Observers can be *many* functions that are told something happened; they react on the side, and the fit comes out the same whether there are zero of them or ten. (Early stopping is the borderline case, since it can end the loop. Keras handles that the same way, with a flag the callback sets.)

**Functions or classes?** The lecture example used plain functions as listeners. Here most of them are small classes, because each keeps a little state between calls (the history so far, the best loss, how many bad steps). That is the "a class when state and behaviour belong together" rule from Lecture 20.

**When it is not worth it:** a loop with one `print` in it does not need a callback system. It starts to pay off when the side jobs outnumber the real work, or when different runs need different side jobs (quiet batch runs, chatty interactive ones).

**You have already used this, or will:**

 - Keras: `model.fit(..., callbacks=[EarlyStopping(), ModelCheckpoint(...)])`. My own training script for detecting ship tracks in satellite images used four: TensorBoard logging, checkpoints, reducing the learning rate when the loss stalled, and one I wrote to compute extra metrics.
 - PyTorch Lightning callbacks, and the `logging` module, where handlers are listeners for log messages.
 - `scipy.optimize.minimize(..., callback=...)`, which runs below.

In [ ]:
from scipy.optimize import minimize

seen = []
result = minimize(lambda p: np.mean((p[0] * x + p[1] - y) ** 2), x0=[0.0, 0.0],
                  callback=lambda p: seen.append(p.copy()))      # called after every iteration
print("scipy:", result.x.round(3), "after", len(seen), "iterations")

### Exercise 3 - stop when it blows up

With too large a learning rate, gradient descent diverges and the loss grows until it becomes `inf`. Write a `StopOnDivergence` callback that raises `Stop` as soon as the loss is not finite, or more than 1000 times the first loss it saw. Run `fit_tips(x, y, lr=1.5, callbacks=[watcher, StopOnDivergence()])` with a `History` called `watcher`, and check that it stops within the first 50 steps.

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
class StopOnDivergence:
    def __init__(self, factor=1000):
        self.factor, self.first = factor, None
    def __call__(self, step, loss, a, b):
        if self.first is None:
            self.first = loss
        if not np.isfinite(loss) or loss > self.factor * self.first:
            raise Stop(f"loss diverged ({loss:.3g})")

watcher = History()
fit_tips(x, y, lr=1.5, callbacks=[watcher, StopOnDivergence()])
print("steps run:", len(watcher.losses))
assert len(watcher.losses) < 50
print("check passed")

# Case 4. Pipeline: the notebook that got copied

This is the most common one in my own code. You write an analysis notebook, it works, and then you need the same thing for a different variable, region or experiment. The quickest route is to copy the notebook and edit it. I have pairs of notebooks that differ only in which variable they load, and the second one still uses the first one's variable names. I also have one set of three notebooks, about 530 lines each, that differ in 41 lines.

Here the analysis is "how much does a region warm between 1995-2014 and 2081-2100?"

## Before

Two cells, standing in for two notebooks. The second is a copy of the first, edited for the Antarctic.

In [ ]:
# ---- arctic_warming.ipynb ----
arctic = tas.sel(lat=slice(60, 90))
arctic_early = arctic.sel(time=slice(1995, 2014)).mean("time")
arctic_late = arctic.sel(time=slice(2081, 2100)).mean("time")
arctic_change = arctic_late - arctic_early
w = np.cos(np.deg2rad(arctic_change.lat))
arctic_mean = float(arctic_change.weighted(w).mean())
arctic_max = float(arctic_change.max())
arctic_zonal = arctic_change.mean("lon")
print(f"Arctic: mean warming {arctic_mean:.2f} °C, max {arctic_max:.2f} °C, "
      f"warmest latitude {float(arctic_zonal.idxmax()):.1f}")

# ---- antarctic_warming.ipynb (a copy of the above, edited) ----
arctic = tas.sel(lat=slice(-90, -60))
arctic_early = arctic.sel(time=slice(1995, 2014)).mean("time")
arctic_late = arctic.sel(time=slice(2081, 2100)).mean("time")
arctic_change = arctic_late - arctic_early
arctic_mean = float(arctic_change.mean())
arctic_max = float(arctic_change.max())
arctic_zonal = arctic_change.mean("lon")
print(f"Arctic: mean warming {arctic_mean:.2f} °C, max {arctic_max:.2f} °C, "
      f"warmest latitude {float(arctic_zonal.idxmax()):.1f}")

Both lines say "Arctic", because the label was not updated in the copy. That one is easy to spot. The harder one to spot is that the weights line got lost in the edit, so the Antarctic mean is unweighted. It treats the tiny grid cells next to the pole as if they were as big as the ones at 60°S, and comes out about 30% too high. Nothing errors, and 1.08°C looks like a perfectly reasonable number.

The same thing happens with helper functions. When I searched my own research code, a `global_mean` function was defined in 29 different files, and they did not all agree (`lat` or `latitude`, how the weights were named, whether pressure levels were handled).

## The change request

 1. "Do the same for the tropics."
 2. "Use 1851-1900 as the early period instead."

That is a third copy, and then the same edit in three places. Every bug fix has to be found and made in every copy too.

## After

The analysis is a fixed sequence of steps, so we write each step once as a small, general function and compose them. This uses the same `pipeline` helper as Lecture 20.

In [ ]:
def pipeline(*stages):
    # Compose stage functions: run them left to right.
    def run(data):
        for stage in stages:
            data = stage(data)
        return data
    return run

def select(region):
    return lambda da: da.sel(**region)

def change_between(early, late):
    return lambda da: (da.sel(time=slice(*late)).mean("time")
                       - da.sel(time=slice(*early)).mean("time"))

def summarise(da):
    return {"mean": float(area_mean(da)), "max": float(da.max()),
            "warmest lat": float(da.mean("lon").idxmax())}

REGIONS = {"Arctic": {"lat": slice(60, 90)}, "Antarctic": {"lat": slice(-90, -60)}}
EARLY, LATE = (1995, 2014), (2081, 2100)

def warming_table(regions, early, late):
    rows = {name: pipeline(select(region), change_between(early, late), summarise)(tas)
            for name, region in regions.items()}
    return pd.DataFrame(rows).T.round(2)

print(warming_table(REGIONS, EARLY, LATE))

The Arctic numbers match the "before" version. The Antarctic mean drops from 1.08°C to 0.82°C, because it is now area-weighted like the Arctic one.

Because each stage is small and general, we can test it on a tiny made-up grid where we can work out the answer by hand. Two grid cells, one at the equator (weight 1) and one at 60°N (weight 0.5):

In [ ]:
tiny = xr.DataArray([[1.0], [3.0]], dims=["lat", "lon"], coords={"lat": [0.0, 60.0], "lon": [0.0]})
print(float(area_mean(tiny)))
assert np.isclose(float(area_mean(tiny)), (1 * 1.0 + 3 * 0.5) / 1.5)

## The change again

A new region is one line in `REGIONS`, and a new period is one argument:

In [ ]:
REGIONS["Tropics"] = {"lat": slice(-23.5, 23.5)}
print(warming_table(REGIONS, early=(1851, 1900), late=LATE))

## Discussion

**What makes it a pipeline rather than just "some functions"?** Each stage takes the output of the one before and knows nothing about the others, so you can reorder, swap, test, cache or re-run stages one at a time. None of the stages mention the Arctic or a particular year, which is why the same pieces work for any region and period. In Lecture 20 we did the same thing across two different datasets.

**When it is not worth it:** a one-off look at the data does not need this. The time to do it is the moment you are about to copy a notebook to change one thing. Copying is fine for a quick check, but once two copies both matter, they will drift apart.

**You have already used this, or will:**

 - `df.pipe(clean).pipe(add_rate).pipe(summarise)`, and `ds.pipe(...)` in xarray.
 - `sklearn.pipeline.Pipeline([("scale", StandardScaler()), ("model", Ridge())])`.
 - Workflow tools like Snakemake and Airflow, which are pipelines of whole scripts, arranged as the DAGs from Lecture 18.
 - Unix pipes: `cat trips.csv | grep JFK | wc -l`.

### Exercise 4 - a pipeline for the taxi data

Using the same `pipeline` helper, build `zone_tips`, a pipeline of three stages for the taxi data:

 1. keep card payments only (`payment_type == 1`) with `fare_amount` of at least 2.5,
 2. add a `tip_pct` column, 100 times tip over fare,
 3. return the mean `tip_pct` for each pickup zone (`pulocationid`) that has at least 50 trips, highest first.

Write each stage so it takes and returns a DataFrame. Then store `zone_tips(taxi)` as `result`.

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
def card_trips(df):
    return df[(df["payment_type"] == 1) & (df["fare_amount"] >= 2.5)]

def add_tip_pct(df):
    return df.assign(tip_pct=100 * df["tip_amount"] / df["fare_amount"])

def mean_by_zone(df, min_trips=50):
    g = df.groupby("pulocationid")["tip_pct"]
    return g.mean()[g.size() >= min_trips].sort_values(ascending=False)

zone_tips = pipeline(card_trips, add_tip_pct, mean_by_zone)
result = zone_tips(taxi)
print(result.round(1).head())

check = taxi.query("payment_type == 1 and fare_amount >= 2.5")
check = (100 * check["tip_amount"] / check["fare_amount"]).groupby(check["pulocationid"])
expected = check.mean()[check.size() >= 50]
assert np.allclose(result.sort_index(), expected.sort_index())
assert result.is_monotonic_decreasing
print("check passed")

# Case 5. Decorator: the cache that went stale

Some steps are slow: regridding model output, reading thousands of files, fitting something at every grid point. The usual fix is to save the result the first time and load it after that. My version of this was in a notebook comparing aerosol models with satellite data, where regridding took long enough that I cached it to a file. That `if` file exists `else` compute-and-save block was copied for each satellite product.

Here the slow step is a warming trend at every grid point. It is actually fast, so `time.sleep(1)` stands in for the slow part.

## Before

In [ ]:
def warming_trend(start, end):
    time.sleep(1)                                   # pretend this takes 20 minutes
    fit = tas.sel(time=slice(start, end)).polyfit("time", deg=1)
    return fit.polyfit_coefficients.sel(degree=1) * 10      # °C per decade

cache_file = tmp / "trend.nc"
if cache_file.exists():
    trend = xr.load_dataarray(cache_file)
else:
    trend = warming_trend(1971, 2020)
    trend.to_netcdf(cache_file)
print(f"1971-2020 trend: {float(area_mean(trend)):.3f} °C per decade")

## The change request

"Can you use 1991-2020 instead?" So we change the years and re-run:

In [ ]:
if cache_file.exists():
    trend = xr.load_dataarray(cache_file)
else:
    trend = warming_trend(1991, 2020)
    trend.to_netcdf(cache_file)
print(f"1991-2020 trend: {float(area_mean(trend)):.3f} °C per decade")
print(f"actual 1991-2020 trend: {float(area_mean(warming_trend(1991, 2020))):.3f} °C per decade")

It ran instantly and gave the 1971-2020 number again, because the file was already there and the cache has no idea the years changed. Nothing errors, you just get the old answer. The copy of my notebook starts with a cell that deletes the cache files by hand, which is how I found out.

The thing that varies here is nothing about the calculation. Caching is extra behaviour we want *around* a function, keyed on its arguments, for any slow function. That is what a **decorator** is for: wrap a function, keep its interface, add behaviour.

## After

In [ ]:
def disk_cache(fn):
    # Save fn's result to a file named after the function and its arguments.
    @functools.wraps(fn)
    def wrapper(*args):
        key = "_".join([fn.__name__, *map(str, args)])
        path = tmp / f"{key}.nc"
        if path.exists():
            print(f"  loaded {path.name}")
            return xr.load_dataarray(path)
        result = fn(*args)
        result.to_netcdf(path)
        print(f"  computed and saved {path.name}")
        return result
    return wrapper

@disk_cache
def warming_trend(start, end):
    time.sleep(1)
    fit = tas.sel(time=slice(start, end)).polyfit("time", deg=1)
    return fit.polyfit_coefficients.sel(degree=1) * 10

for years in [(1971, 2020), (1991, 2020), (1971, 2020)]:
    t0 = time.perf_counter()
    trend = warming_trend(*years)
    print(f"{years}: {float(area_mean(trend)):.3f} °C per decade, {time.perf_counter() - t0:.2f} s")

## The change again

Changing the years now just works: a new set of arguments is a new file, and the old result is still there if you go back to it. Caching a second slow function is one line (`@disk_cache` above it), instead of another copy of the `if`/`else` block.

## Discussion

**Why Decorator and not Adapter?** Both wrap a function. An adapter changes the interface so something fits (Case 1). A decorator keeps the interface exactly the same, so callers cannot tell, and adds behaviour around it: caching, timing, retrying, logging, checking inputs.

Note, this cache still has a gap. If you change the *code* inside `warming_trend`, the arguments are the same so it loads the old file. `joblib.Memory` handles that by also keying on the function's source code, and it is what I would use for real work.

**When it is not worth it:** if a step takes a few seconds, do not cache it; the stale-cache bugs cost more than the time saved.

**You have already used this:**

 - `@functools.lru_cache`, an in-memory cache, from the lecture.
 - `@joblib.Memory(...).cache`, the grown-up version of `disk_cache`.
 - `@st.cache_data` in Streamlit dashboards, and `@numba.njit`, which keeps the interface but compiles the function.
 - `@pytest.mark.parametrize`, coming up in the testing lecture.

### Exercise 5 - keyword arguments

`disk_cache` only accepts positional arguments, so `warming_trend(start=1981, end=2010)` fails. Write `disk_cache_kw`, a version that also accepts keyword arguments and includes them in the cache key. Then check that calling a decorated function twice with the same keyword arguments only computes once. (Use the `calls` list below to count how often the body runs.)

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
def disk_cache_kw(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        parts = [fn.__name__, *map(str, args)] + [f"{k}={v}" for k, v in sorted(kwargs.items())]
        path = tmp / f"{'_'.join(parts)}.nc"
        if path.exists():
            return xr.load_dataarray(path)
        result = fn(*args, **kwargs)
        result.to_netcdf(path)
        return result
    return wrapper

calls = []

@disk_cache_kw
def trend_kw(start, end):
    calls.append((start, end))
    fit = tas.sel(time=slice(start, end)).polyfit("time", deg=1)
    return fit.polyfit_coefficients.sel(degree=1) * 10

first = trend_kw(start=1981, end=2010)
second = trend_kw(start=1981, end=2010)
print("body ran", len(calls), "time(s)")
assert len(calls) == 1
assert np.allclose(first, second)
print("check passed")

# Spot the pattern

To finish, some short snippets, mostly from libraries you have used. For each, name the pattern: **Strategy, Factory, Adapter, Decorator, Observer, Pipeline, Facade**, or **none** if a plain function is the right answer. A few are from my own code.

 1. `sorted(trips, key=lambda t: t["distance"])`
 2. `scipy.optimize.minimize(f, x0, callback=print_progress)`
 3. `xr.open_dataset("run.nc", engine="h5netcdf")`
 4. `pd.read_csv(io.StringIO(text))`, where `StringIO` makes a string behave like a file
 5. `df.pipe(clean).pipe(add_rate).pipe(summarise)`
 6. `plt.plot(x, y)`, which creates the figure, axes and line objects for you behind one call
 7. `logging.getLogger().addHandler(logging.FileHandler("run.log"))`
 8. `@functools.lru_cache` above a slow function
 9. `def c_to_f(c): return c * 9 / 5 + 32`
 10. From my code: a base class `Deseasonaliser` with subclasses `AggDeseasonaliser` and `SinefitDeseasonaliser`, which share `fit_cycle()` and `evaluate_cycle()` so the analysis can use either. (There is a flaw: the subclasses take different arguments to `fit_cycle`. Why is that a problem?)
 11. From my code: `@ungridded_dataframe_wrapper`, which lets a function written for pandas DataFrames be called on my own ungridded-data type by converting it first.
 12. From my code: an `@initializer` decorator, copied from StackOverflow, that inspects `__init__` and sets every argument as an attribute automatically.

Write your answers in the empty cell as a dictionary, `my_answers = {1: "strategy", 2: ..., ...}`, then run the solution.

In [ ]:
# Your turn. Try it here, then run the solution cell below to check.

In [ ]:
# --- solution ---
answers = {
    1: "strategy",   # the sort is fixed, the key is swappable
    2: "observer",   # told after every iteration, does not change the result
    3: "factory",    # picks and sets up a backend from a name
    4: "adapter",    # makes a string fit the "file" interface read_csv expects
    5: "pipeline",
    6: "facade",     # one simple call in front of a lot of machinery
    7: "observer",   # handlers listen for log messages
    8: "decorator",
    9: "none",       # a plain function is exactly right
    10: "strategy",  # interchangeable deseasonalisers; but different arguments break "interchangeable"
    11: "adapter",   # written as a decorator, but its job is to make a type fit
    12: "decorator", # and over-engineered: a @dataclass does this, readably
}
if "my_answers" in globals():
    right = [k for k in answers if my_answers.get(k, "").lower() == answers[k]]
    print(f"{len(right)} / {len(answers)} right; check:", [k for k in answers if k not in right])
for k, v in answers.items():
    print(k, v)

A few notes on the trickier ones:

 - **4 and 11** are adapters, even though 11 is written with `@`. Ask what the wrapper is *for*: if it changes the interface so something fits, it is an adapter; if it keeps the interface and adds behaviour, it is a decorator.
 - **10** is Strategy by subclassing, the classic object-oriented way to write it. The flaw matters because the whole point is that the analysis can use either one without knowing which. If the two take different arguments, the calling code needs an `if` on the type again, which is what we were trying to get rid of.
 - **12** works, but it is magic that the next reader has to look up, and it broke when Python removed `inspect.getargspec`. `@dataclass` does the same job and everyone knows it. This is the over-engineering from the end of Lecture 20.
 - **9** is there because "none" is often the right answer.

# Done

If you want to practise this with an agent, try the experiment from `practice/working_with_agents.md` (the free claude.ai plan is enough). Paste in one of the "before" scripts and ask for it to be cleaned up, once without naming a pattern and once naming it ("refactor the metrics into a strategy"). Compare what you get back. Naming the pattern usually gets you closer to what you wanted in one go, which is the point of having the vocabulary.